In [11]:
!pip -q install pyreadstat

import pandas as pd

df = pd.read_stata("/content/drive/MyDrive/Colab Notebooks/Business/ReplicationData2.dta")
df.info()
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 670 entries, 0 to 669
Data columns (total 20 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   gpa                    525 non-null    float64
 1   cum_total_crds         632 non-null    float64
 2   gender                 670 non-null    int16  
 3   asian                  500 non-null    float64
 4   black                  500 non-null    float64
 5   hawaiian               500 non-null    float64
 6   hispanic               500 non-null    float64
 7   unknown                500 non-null    float64
 8   white                  500 non-null    float64
 9   ethnic_dummy           507 non-null    float32
 10  format_ol              670 non-null    float32
 11  format_blended         670 non-null    float32
 12  sat_math_NEW           611 non-null    float64
 13  sat_verbal_NEW         612 non-null    float64
 14  enroll_count           519 non-null    float64
 15  format

,gpa,cum_total_crds,gender,asian,black,hawaiian,hispanic,unknown,white,ethnic_dummy,format_ol,format_blended,sat_math_NEW,sat_verbal_NEW,enroll_count,format_f2f_v_ol,format_f2f_v_blended,format_combined_v_f2f,falsexam,experiment1
0,2.014,63.0,1,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,540.0,540.0,NaN,0.0,0.0,1.0,0.0,0.0
1,3.720,33.0,1,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,590.0,630.0,NaN,0.0,0.0,1.0,0.0,0.0
2,NaN,4.0,0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,650.0,570.0,NaN,0.0,0.0,1.0,0.0,0.0
3,NaN,10.0,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,690.0,690.0,NaN,0.0,0.0,1.0,0.0,0.0
4,NaN,0.0,0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,480.0,420.0,NaN,0.0,0.0,1.0,0.0,0.0


In [12]:
sample=df.query("enroll_count==3")
#sample.info()
online=sample.query("format_ol==1").copy()
f2f=sample.query("format_ol!=1 and format_blended!=1").copy()
ATE=online['falsexam'].mean()-f2f['falsexam'].mean()
print(ATE)

-4.9122314


In [13]:
from scipy import stats
t_stat, p_val=stats.ttest_ind(online['falsexam'],f2f['falsexam'])
print(f"T-statistic: {t_stat:.4f}, P-value:{p_val:.4f}")

T-statistic: -2.9247, P-value:0.0038


Calculate the learning mode

Mean


T-Test T-Stat abd P-Value
Online vs Blended
Online vs F2F
Blended vs F2F


In [14]:
from scipy import stats

sample = df.query("enroll_count == 3").copy()

online = sample[sample["format_ol"] == 1]["falsexam"].dropna()
blended = sample[sample["format_blended"] == 1]["falsexam"].dropna()
f2f = sample[(sample["format_ol"] == 0) & (sample["format_blended"] == 0)]["falsexam"].dropna()

print("Means")
print("Online:", online.mean())
print("Blended:", blended.mean())
print("F2F:", f2f.mean())
print()

t1 = stats.ttest_ind(online, blended, equal_var=False)
print("Online vs Blended")
print("T-stat:", t1.statistic)
print("P-value:", t1.pvalue)
print()

t2 = stats.ttest_ind(online, f2f, equal_var=False)
print("Online vs F2F")
print("T-stat:", t2.statistic)
print("P-value:", t2.pvalue)
print()

t3 = stats.ttest_ind(blended, f2f, equal_var=False)
print("Blended vs F2F")
print("T-stat:", t3.statistic)
print("P-value:", t3.pvalue)

Means
Online: 73.635254
Blended: 77.093735
F2F: 78.547485

Online vs Blended
T-stat: -1.9049511492731843
P-value: 0.0585856480025066

Online vs F2F
T-stat: -2.7792867897293885
P-value: 0.006143692819092313

Blended vs F2F
T-stat: -1.1168635439492236
P-value: 0.2652600684046152


In [16]:
import statsmodels.formula.api as smf
mod1='falsexam~format_ol+format_blended'
reg1=smf.ols(mod1,data=sample).fit()
print(reg1.summary())

                            OLS Regression Results                            
Dep. Variable:               falsexam   R-squared:                       0.030
Model:                            OLS   Adj. R-squared:                  0.024
Method:                 Least Squares   F-statistic:                     4.922
Date:                Tue, 14 Apr 2026   Prob (F-statistic):            0.00785
Time:                        04:32:31   Log-Likelihood:                -1246.4
No. Observations:                 323   AIC:                             2499.
Df Residuals:                     320   BIC:                             2510.
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
Intercept         78.5475      1.052     74.

In [24]:
control1='gpa+cum_total_crds+sat_math_NEW+sat_verbal_NEW'
mod2=f'falsexam~format_ol+format_blended+{control1}'
reg2=smf.ols(mod2,data=sample).fit()
print(reg2.summary())

                            OLS Regression Results                            
Dep. Variable:               falsexam   R-squared:                       0.234
Model:                            OLS   Adj. R-squared:                  0.216
Method:                 Least Squares   F-statistic:                     13.11
Date:                Tue, 14 Apr 2026   Prob (F-statistic):           6.09e-13
Time:                        05:06:45   Log-Likelihood:                -990.75
No. Observations:                 265   AIC:                             1995.
Df Residuals:                     258   BIC:                             2021.
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
Intercept         41.1684      7.429      5.

In [26]:
control2='gpa+cum_total_crds+sat_math_NEW+sat_verbal_NEW+gender+asian+black+hawaiian+unknown+white+hispanic'
mod2=f'falsexam~format_ol+format_blended+{control2}'
reg3=smf.ols(mod2,data=sample).fit()
print(reg3.summary())

                            OLS Regression Results                            
Dep. Variable:               falsexam   R-squared:                       0.345
Model:                            OLS   Adj. R-squared:                  0.307
Method:                 Least Squares   F-statistic:                     8.883
Date:                Tue, 14 Apr 2026   Prob (F-statistic):           1.42e-13
Time:                        05:10:27   Log-Likelihood:                -774.31
No. Observations:                 215   AIC:                             1575.
Df Residuals:                     202   BIC:                             1618.
Df Model:                          12                                         
Covariance Type:            nonrobust                                         
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
Intercept         33.0553      6.992      4.